Demo & Benchmarks
=================

### Setup & Imports

In [1]:
import gc
import glob
import os
import time
import tracemalloc

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import io

from batch_numba import SpectralReconNumba as sr

### Reconstruction setup

In [2]:
CROP = (500, 2500, 1000, 2500)  # ROI indexes
SPECTRAL_BAND_WIDTH = 100       # size of the spectral channell, size in lambda dim of the output image (should be greater than actual expected band distance)
CUSTOM_LINES_NUM = True         # on/off automatic lines num estimation
LINES_NUM = 22                  # num of fixed lines/bands (~20-22 lines per 2000 px)
CUSTOM_DIST = False             # on/off automatic lines/bands width estimation
DIST_UP = 35                    # custom dist from line to the lower index border (up line in imshow output)
DIST_DOWN = 45                  # custom dist from line to the upper index border (down line in imshow output)
DIST_OFFSET = -15               # line border idx offset in px, may be negative
MASK_WIDTH = 80                 # minimal distance between detected lines
PREC_ALLOCATION = True          #

DATA_DIR  = 'data/QD_mix_40_phase'
all_paths = sorted(glob.glob(f'{DATA_DIR}/*.tiff'))
print(f'Found {len(all_paths)} images in {DATA_DIR}')

analyzer_opt = sr(crop=CROP,
    spectral_band_width=SPECTRAL_BAND_WIDTH,
    custom_lines_num=CUSTOM_LINES_NUM,
    lines_num=LINES_NUM,
    dist_offset=DIST_OFFSET,
    custom_dist=CUSTOM_DIST,
    dist_up=DIST_UP,
    dist_down=DIST_DOWN,
    mask_width=MASK_WIDTH,
    precise_allocation = PREC_ALLOCATION)

Found 40 images in data/QD_mix_40_phase


---

# Single-Image Fast Test

In [ ]:
sample_path = all_paths[3]
print(f'Sample image: {sample_path}')

raw_img = analyzer_opt.load_image(sample_path)

In [ ]:
t0 = time.perf_counter()
single_result = analyzer_opt.process_single(sample_path)
elapsed = time.perf_counter() - t0

print(f'process_single() completed in {elapsed:.2f} s')
print(f'  Compact bands: {single_result.spectral_bands.shape}')
print(f'  Row indices:   {single_result.row_indices}')
print(f'  Image shape:   {single_result.image_shape}')
print(f'  Memory (bands): {single_result.spectral_bands.nbytes / (1024**2):.2f} MB')

In [ ]:
single_result.plot_reg_structure(cmap='grey')

In [ ]:
single_spectral_img = single_result.spectral_img
print(single_spectral_img.shape)

single_mip = np.max(single_spectral_img, axis=2)

fig, axes = plt.subplots(1, 2, figsize=(20, 15))
axes[0].imshow(raw_img, cmap='jet')
axes[0].set_title('Raw (cropped)')
axes[1].imshow(single_mip, cmap='jet')
axes[1].set_title('Recon MIP')
plt.tight_layout()
plt.show()

---

# Single-Image Pipeline Walkthrough

In [ ]:
sample_path = all_paths[3]
print(f'Sample image: {sample_path}')

### Load image & edge filtering

In [ ]:
raw_img = analyzer_opt.load_image(sample_path)
edges   = ndi.prewitt(raw_img, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(raw_img, cmap='gray')
axes[0].set_title('Raw (cropped)')
axes[1].imshow(edges, cmap='gray')
axes[1].set_title('Prewitt edge')
plt.tight_layout()
plt.show()

### 2.2 Detect & regularize structures

In [ ]:
structures = analyzer_opt.detect_structures(edges)
reg_lines  = analyzer_opt.regularize_structures(structures)

print(f"Light lines: {reg_lines['light'].shape}")
print(f"dist_up  = {reg_lines['params']['dist_up']:.1f} px")
print(f"dist_down = {reg_lines['params']['dist_down']:.1f} px")
print(f"offset = {reg_lines['params']['offset']:.1f} px")

### 2.3 Extract spectral bands & compact allocation

In [ ]:
bands = analyzer_opt.extract_spectral_bands(
    raw_img, reg_lines, SPECTRAL_BAND_WIDTH
)
print(f'Spectral bands shape: {bands.shape}  '
      f'(num_bands, image_width, spectral_width)')

row_idx, used_bands = analyzer_opt.allocate_spectral_pixels(
    raw_img, reg_lines, bands
)
print(f'Row indices ({len(row_idx)}): {row_idx}')
print(f'Compact bands shape: {used_bands.shape}')
print(f'Compact bands memory: {used_bands.nbytes / (1024**2):.2f} MB')

### 2.4 Expand to full spectral image (optional)

In [ ]:
full_spectral = analyzer_opt.expand_to_full(
    raw_img.shape[:2], row_idx, used_bands,
)
print(f'Full spectral image: {full_spectral.shape}, '
      f'{full_spectral.nbytes / (1024**2):.1f} MB')

mip_single = np.max(full_spectral, axis=2)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(mip_single, cmap='jet')
ax.set_title('Single-image max-intensity projection (optimized)')
plt.colorbar(ax.images[0], ax=ax)
plt.show()
del full_spectral  # free immediately

---
# Full Batch Reconstruction

In [3]:
batch_paths = all_paths[:40]

t0 = time.perf_counter()
batch_result = analyzer_opt.process_batch(batch_paths, method='max')
batch_time = time.perf_counter() - t0

spectral_batch = batch_result.spectral_img
print(f'\nBatch reconstruction: {batch_time:.1f} s')
print(f'Result shape: {spectral_batch.shape}')
print(f'Images processed: {batch_result.num_images}')

# io.imsave(f'QD_mix_40_phase_prec-alloc_offset{analyzer_opt.dist_offset}.tiff', np.moveaxis(spectral_batch, -1,0))

[1/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0000.tiff ...
Custom lines number: 22
[2/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0001.tiff ...
Custom lines number: 22
[3/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0002.tiff ...
Custom lines number: 22
[4/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0003.tiff ...
Custom lines number: 22
[5/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0004.tiff ...
Custom lines number: 22
[6/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0005.tiff ...
Custom lines number: 22
[7/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0006.tiff ...
Custom lines number: 22
[8/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0007.tiff ...
Custom lines number: 22
[9/40] Processing Basler_boA4096-180cm__40460831__20260320_123758244_0008.tiff ...
Custom lines number: 22
[10/40] Processing Basler_boA4096-180

### Spatial slices at selected spectral channels

In [ ]:
S = spectral_batch.shape[2]
channels = [int(S * f) for f in (0.1, 0.3, 0.5, 0.7, 0.9)]

fig, axes = plt.subplots(1, len(channels), figsize=(20, 6))
for ax, ch in zip(axes, channels):
    ax.imshow(spectral_batch[:, :, ch], cmap='inferno')
    ax.set_title(f'Channel {ch}')
    ax.axis('off')
fig.suptitle('Spatial slices at selected spectral channels', fontsize=14)
plt.tight_layout()
plt.show()

### Max-intensity projection

In [ ]:
mip = np.max(spectral_batch, axis=2)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(mip, cmap='jet', vmin=0)
ax.set_title('Max-intensity projection (optimized batch, method="max")')
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
# interpolation
import numpy as np
from scipy.interpolate import griddata

def interpolate_missing_zeros(image: np.ndarray, method: str = 'linear') -> np.ndarray:
    """
    2D interpolation robust to zeros
    """
    valid_mask = image != 0
    coords = np.array(np.nonzero(valid_mask)).T
    values = image[valid_mask]

    grid_x, grid_y = np.mgrid[0:image.shape[0], 0:image.shape[1]]

    interpolated_image = griddata(points=coords,
                                  values=values,
                                  xi=(grid_x, grid_y),
                                  method=method,
                                  fill_value=0)

    return interpolated_image

In [ ]:
inter_mip = interpolate_missing_zeros(mip)
plt.figure(figsize=(10, 8))
plt.imshow(inter_mip, cmap='jet', vmin=0)
plt.title('Interpolated max-intensity projection')
plt.colorbar()
plt.show()

### Interpolation

In [4]:
t0 = time.perf_counter()
batch_result.interpolate_missing_zeros()
batch_time = time.perf_counter() - t0

print(f'\nBatch interpolated: {batch_time:.1f} s')

batch_result_interpolated = batch_result.spectral_img_interpolated
io.imsave(f'QD_mix_40_phase_prec-alloc_offset{analyzer_opt.dist_offset}_inter.tiff', np.moveaxis(batch_result_interpolated, -1,0))

Starting parallel Numba 2D smooth cubic interpolation...
Interpolation complete.

Batch interpolated: 10.5 s


---
## 6. Row-Index Metadata

The optimized pipeline automatically collects row-index metadata for every
image in the batch. This records which rows in the reconstructed image
received spectral data from each source frame.

In [ ]:
print(f'Number of images with metadata: {len(batch_result.row_indices)}')
print()

# show first 3 entries
for i, (path, indices) in enumerate(batch_result.row_indices.items()):
    if i >= 3:
        print('  ...')
        break
    print(f'  {os.path.basename(path)}:')
    print(f'    row_indices ({len(indices)}): {indices}')

### 6.1 Save metadata to disk (NPZ, JSON, YAML)

In [ ]:
# Save as compressed NumPy archive
batch_result.save_metadata(f'QD_mix_40_phase_prec-alloc_offset{analyzer_opt.dist_offset}.npz')
print('Saved row_indices.npz')

# Save as human-readable JSON
batch_result.save_metadata(f'QD_mix_40_phase_prec-alloc_offset{analyzer_opt.dist_offset}.json')
print('Saved row_indices.json')

# Save as human-readable YAML
batch_result.save_metadata(f'QD_mix_40_phase_prec-alloc_offset{analyzer_opt.dist_offset}.yaml')
print('Saved row_indices.yaml')

### 6.2 Load metadata back & verify consistency

In [ ]:
loaded_npz  = BatchResult.load_metadata('row_indices.npz')
loaded_json = BatchResult.load_metadata('row_indices.json')
loaded_yaml = BatchResult.load_metadata('row_indices.yaml')

print(f'NPZ  entries: {len(loaded_npz)}')
print(f'JSON entries: {len(loaded_json)}')
print(f'YAML entries: {len(loaded_yaml)}')

# verify consistency across all formats
for key in loaded_npz:
    assert np.array_equal(loaded_npz[key], loaded_json[key]), \
        f'NPZ/JSON mismatch for {key}'
    assert np.array_equal(loaded_npz[key], loaded_yaml[key]), \
        f'NPZ/YAML mismatch for {key}'
print('\n✅ NPZ, JSON, and YAML metadata are all consistent!')

### 6.3 Visualize row indices across the batch

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, (path, indices) in enumerate(batch_result.row_indices.items()):
    ax.scatter(indices, [i] * len(indices), s=8, alpha=0.8)

ax.set_xlabel('Row index in reconstructed image')
ax.set_ylabel('Image number in batch')
ax.set_title('Row indices per batch image')
plt.tight_layout()
plt.show()

---
## 7. Reconstruction Benchmarks

### 7.1 Signal-to-Noise Ratio (SNR)

In [ ]:
snr_per_row = []

for r in row_signal:
    row_data = spectral_batch[r, :, :]
    nz_mask = row_data.any(axis=1)
    if nz_mask.sum() == 0:
        continue
    vals = row_data[nz_mask]
    mean_val = vals.mean()
    std_val  = vals.std()
    if std_val > 0:
        snr_per_row.append(mean_val / std_val)

snr_arr = np.array(snr_per_row)
print(f'SNR across {len(snr_arr)} signal rows:')
print(f'  mean = {snr_arr.mean():.2f},  std = {snr_arr.std():.2f}')
print(f'  min  = {snr_arr.min():.2f},  max = {snr_arr.max():.2f}')

plt.figure(figsize=(8, 4))
plt.hist(snr_arr, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel('SNR')
plt.ylabel('Count')
plt.title('SNR distribution across signal rows')
plt.axvline(snr_arr.mean(), color='r', ls='--', label=f'mean={snr_arr.mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()

### 7.2 Band Uniformity

In [ ]:
band_totals = []
for r in row_signal:
    total = spectral_batch[r, :, :].sum()
    if total > 0:
        band_totals.append(total)

band_totals = np.array(band_totals)
cv = band_totals.std() / band_totals.mean() if len(band_totals) > 0 else 0
print(f'Band uniformity: CV = {cv:.4f}  (lower is better)')
print(f'  mean = {band_totals.mean():.0f},  std = {band_totals.std():.0f}')

### 7.3 Cross-Frame Consistency

In [ ]:
n_frames = min(3, len(all_paths))
frame_indices = np.linspace(0, len(all_paths) - 1, n_frames, dtype=int)
frames = {}

for fi in frame_indices:
    sr = analyzer_opt.process_single(all_paths[fi])
    img = analyzer_opt.expand_to_full(
        sr.image_shape, sr.row_indices, sr.spectral_bands
    )
    frames[fi] = img
    del sr

print('Pairwise NRMSE:')
keys = list(frames.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        a, b = frames[keys[i]], frames[keys[j]]
        rmse  = np.sqrt(np.mean((a.astype(float) - b.astype(float))**2))
        denom = max(a.max() - a.min(), 1e-10)
        nrmse = rmse / denom
        print(f'  Frame {keys[i]} vs {keys[j]}: NRMSE = {nrmse:.6f}')

del frames  # free